In [ ]:
#Priyanka Barde Skycargo Analysis

#import necessary libraries
import pandas as pd
import sqlite3 
import os

In [ ]:
# 1. Name database file
db_name = "cargo_data.db"
conn = sqlite3.connect(db_name)

# 2. Path to CSV files (use '.' if they are in the same folder as this notebook)
csv_folder = './shipments_db' 

for filename in os.listdir(csv_folder):
    if filename.endswith(".csv"):
        # Create a table name based on the filename (e.g., 'users.csv' -> 'users')
        table_name = os.path.splitext(filename)[0]
        
        # Load CSV into a DataFrame
        df = pd.read_csv(os.path.join(csv_folder, filename))
        
        # Write to SQLite
        # 'replace' will overwrite existing tables; use 'append' to add data to them
        df.to_sql(table_name, conn, if_exists='replace', index=False)
        
        print(f"✅ Created table '{table_name}' from {filename}")



✅ Created table 'cargo_events' from cargo_events.csv
✅ Created table 'exceptions' from exceptions.csv
✅ Created table 'flights' from flights.csv
✅ Created table 'shipments' from shipments.csv


In [ ]:
# 3. Create a cursor object to execute SQL commands
cursor = conn.cursor()

In [ ]:
# 4. Drop the fact_shipments table if it exists
cursor.execute(''' DROP TABLE IF EXISTS fact_shipments; ''')

In [ ]:
# 5. Create the fact_shipments table with required aggregations
cursor.execute('''
              
CREATE TABLE fact_shipments AS
SELECT
    s.shipment_id,
    s.priority,
    s.origin_hub,
    s.destination_hub,
    s.sla_hours,

    -- Received time: first "Received at Origin" event
    MIN(CASE WHEN e.event_name = 'Received at Origin' THEN e.event_timestamp END) AS received_time,

    -- Delivered time: last "Delivered" event
    MAX(CASE WHEN e.event_name = 'Delivered' THEN e.event_timestamp END) AS delivered_time

FROM shipments s
LEFT JOIN cargo_events e
    ON s.shipment_id = e.shipment_id

GROUP BY s.shipment_id;
 ''')

Calculate all the necessary columns for the fact table using SQLite3

In [ ]:
# 6. Add actual_transit_hours and shipment_status columns
cursor.execute(''' 
ALTER TABLE fact_shipments
ADD COLUMN actual_transit_hours REAL;
 ''')

cursor.execute(''' 
ALTER TABLE fact_shipments
ADD COLUMN shipment_status TEXT;
 ''')

cursor.execute(''' -- Update actual_transit_hours
UPDATE fact_shipments
SET actual_transit_hours = (julianday(delivered_time) - julianday(received_time)) * 24
WHERE delivered_time IS NOT NULL;
''')

cursor.execute(''' -- Update shipment_status
UPDATE fact_shipments
SET shipment_status = 
    CASE
        WHEN received_time IS NULL THEN 'Data Issue'
        WHEN delivered_time IS NOT NULL AND actual_transit_hours <= sla_hours THEN 'Delivered On-Time'
        WHEN delivered_time IS NOT NULL AND actual_transit_hours > sla_hours THEN 'Delivered Late'
        WHEN delivered_time IS NULL AND (julianday('2025-02-01 00:00:00') - julianday(received_time)) * 24 <= sla_hours * 0.7 THEN 'In Transit – Within SLA'
        WHEN delivered_time IS NULL THEN 'In Transit – At Risk'
        ELSE 'Unknown'
    END; ''')

In [ ]:
# 7. Add total_delay_minutes column
#Total minutes is the time shipment was delayed beyond SLA. (If delivered early/on-time → 0)
cursor.execute(''' 
ALTER TABLE fact_shipments
ADD COLUMN total_delay_minutes REAL;
 ''')

cursor.execute(''' 
    UPDATE fact_shipments
    SET total_delay_minutes = 
    CASE
        WHEN delivered_time IS NOT NULL THEN 
            MAX((julianday(delivered_time) - julianday(received_time)) * 24 * 60 - sla_hours * 60, 0)
        ELSE 0
    END
 ''')

In [ ]:
# 8. Add num_delay_events_all column
cursor.execute(''' 
ALTER TABLE fact_shipments
ADD COLUMN num_delay_events_all INTEGER;
 ''')


cursor.execute(''' UPDATE fact_shipments
SET num_delay_events_all = 
    (SELECT COUNT(*)
FROM exceptions
WHERE impact_minutes > 0 AND exceptions.shipment_id = fact_shipments.shipment_id
GROUP BY shipment_id);''')

In [ ]:
# 9. Add num_delay_events_causing_late column
cursor.execute(''' 
ALTER TABLE fact_shipments
ADD COLUMN num_delay_events_causing_late INTEGER;
 ''')


cursor.execute(''' UPDATE fact_shipments
SET num_delay_events_causing_late = 
    (SELECT COUNT(*)
FROM exceptions
WHERE impact_minutes > 0 AND exceptions.shipment_id = fact_shipments.shipment_id 
               AND fact_shipments.shipment_status = 'Delivered Late'
GROUP BY shipment_id);''')

In [11]:
cursor.execute(''' 
ALTER TABLE fact_shipments
ADD COLUMN primary_exception_reason TEXT;
 ''')

In [ ]:
# 10. Add primary_exception_reason column
#The primary exception reason is defined as the
#exception type with the largest delay_minutes

cursor.execute(''' UPDATE fact_shipments
SET primary_exception_reason = 
    (SELECT exception_reason
FROM exceptions e1
WHERE impact_minutes = (
    SELECT MAX(impact_minutes)
    FROM exceptions e2
    WHERE e2.shipment_id = e1.shipment_id)
);''')

In [13]:
cursor.execute(''' 
ALTER TABLE fact_shipments
ADD COLUMN max_delay_hub TEXT;
 ''')

In [ ]:
#   11. Add max_delay_hub column
#The hub where the maximum delay-causing exception occurred for that shipment
cursor.execute(''' UPDATE fact_shipments
SET max_delay_hub = 
    (SELECT hub
FROM cargo_events e1
WHERE shipment_id = (
    SELECT shipment_id
    FROM exceptions e2
    WHERE e2.shipment_id = e1.shipment_id
    ORDER BY e2.impact_minutes DESC
    LIMIT 1)
);''')

In [ ]:
# 12. Add num_events column
# count of events per shipment
cursor.execute(''' 
ALTER TABLE fact_shipments
ADD COLUMN num_events INTEGER;
 ''')

In [16]:
cursor.execute(''' UPDATE fact_shipments
SET num_events = 
    (SELECT COUNT(*)
    FROM cargo_events ce
    WHERE ce.shipment_id = fact_shipments.shipment_id
);''')

In [ ]:
# Commit changes and close the connection
sql_query = 'select * from fact_shipments limit 5'
df = pd.read_sql_query(sql_query, conn)

print(df)

  shipment_id  priority origin_hub destination_hub  sla_hours  \
0   SHP000001   Express        AMS             DXB         48   
1   SHP000002  Standard        LHR             JFK         96   
2   SHP000003  Standard        DXB             JFK         96   
3   SHP000004  Standard        DXB             LHR         96   
4   SHP000005  Standard        DXB             FRA         96   

         received_time       delivered_time  actual_transit_hours  \
0  2025-01-15 14:00:00  2025-01-21 22:00:00                 152.0   
1                 None  2025-01-24 20:00:00                   NaN   
2                 None                 None                   NaN   
3  2025-01-04 10:00:00  2025-01-07 06:00:00                  68.0   
4                 None  2025-02-05 21:00:00                   NaN   

     shipment_status  total_delay_minutes  num_delay_events_all  \
0     Delivered Late               6240.0                   NaN   
1         Data Issue                  NaN                   

In [ ]:
# Export the final fact_shipments table to a CSV file
df2 = pd.read_sql("SELECT * FROM fact_shipments", conn)

df2.to_csv("skycargo_fact.csv", index=False)